# 08 — Batch inference: score the season through the champion

Load the model by its **`@champion` alias** (no version pinning — promote a new champion in
notebook `07` and this notebook automatically uses it) and score every game of the held-out
season. We write predictions back to a gold table and close with three checks:

1. **Accuracy** vs the home-court baseline.
2. **Calibration by decile** — do games we call "70%" actually win ~70% of the time?
3. **Biggest upsets we called right** — games where the model confidently backed the road
   team and was correct.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
from datetime import datetime, timezone
import os
import pandas as pd

import mlflow
from mlflow import MlflowClient

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
FEATURES      = f"{UC_CATALOG}.{GOLD_SCHEMA}.game_features"
GAMES         = f"{UC_CATALOG}.{SILVER_SCHEMA}.games"
PRED          = f"{UC_CATALOG}.{GOLD_SCHEMA}.game_predictions"
MODEL_NAME    = os.getenv("MODEL_NAME", "home_win_classifier")
FULL_MODEL    = f"{UC_CATALOG}.{GOLD_SCHEMA}.{MODEL_NAME}"
HOLDOUT       = int(os.getenv("NBA_HOLDOUT_SEASON", "2025"))

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

FEATURE_COLS = [
    "win_pct_diff", "margin_diff", "net_rtg_diff",
    "efg_for_diff", "tov_for_diff", "orb_for_diff", "ftr_for_diff",
    "efg_against_diff", "tov_against_diff", "drb_against_diff", "ftr_against_diff",
    "rest_diff",
]
print("Model:", f"models:/{FULL_MODEL}@champion")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Model: models:/alexander_booth.basketball_reference_waf_gold.home_win_classifier@champion


## Load the champion and score the holdout season

In [2]:
client = MlflowClient()
champ = client.get_model_version_by_alias(FULL_MODEL, "champion")
model = mlflow.pyfunc.load_model(f"models:/{FULL_MODEL}@champion")
print(f"Loaded champion v{champ.version}")

pdf = spark.table(FEATURES).toPandas()
hold = pdf[pdf["season"] == HOLDOUT].copy()
proba = model.predict(hold[FEATURE_COLS].astype(float))
proba = proba.values.reshape(-1) if hasattr(proba, "values") else proba

pred = hold[["game_sk", "season", "game_date"]].copy()
pred["pred_home_win_prob"] = proba.astype(float)
pred["predicted_home_win"] = (pred["pred_home_win_prob"] >= 0.5).astype(int)
pred["actual_home_win"]    = hold["home_win"].astype(int).values
pred["correct"]            = (pred["predicted_home_win"] == pred["actual_home_win"]).astype(int)
pred["model_version"]      = int(champ.version)
pred["model_alias"]        = "champion"
pred["scored_at_utc"]      = datetime.now(timezone.utc)
pred["batch_id"]           = "holdout_backfill"
pred["game_date"]          = pred["game_date"].astype(str)

(spark.createDataFrame(pred)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(PRED))
print(f"Wrote {len(pred)} predictions to {PRED}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded champion v1


Wrote 1075 predictions to alexander_booth.basketball_reference_waf_gold.game_predictions


## 1) Accuracy vs the home-court baseline

In [3]:
spark.sql(f"""
    SELECT
        COUNT(*) AS games,
        ROUND(AVG(correct), 3) AS model_accuracy,
        ROUND(AVG(actual_home_win), 3) AS home_court_baseline,
        ROUND(AVG(correct) - GREATEST(AVG(actual_home_win), 1 - AVG(actual_home_win)), 3) AS lift_vs_baseline
    FROM {PRED}
""").show()

+-----+--------------+-------------------+----------------+
|games|model_accuracy|home_court_baseline|lift_vs_baseline|
+-----+--------------+-------------------+----------------+
| 1075|         0.658|              0.542|           0.115|
+-----+--------------+-------------------+----------------+



## 2) Calibration by predicted decile

In [4]:
spark.sql(f"""
    WITH b AS (
        SELECT *, LEAST(9, CAST(FLOOR(pred_home_win_prob * 10) AS INT)) AS decile
        FROM {PRED}
    )
    SELECT decile,
           COUNT(*) AS games,
           ROUND(AVG(pred_home_win_prob), 3) AS avg_predicted,
           ROUND(AVG(actual_home_win), 3)    AS actual_home_win_rate
    FROM b GROUP BY decile ORDER BY decile
""").show()

+------+-----+-------------+--------------------+
|decile|games|avg_predicted|actual_home_win_rate|
+------+-----+-------------+--------------------+
|     0|    5|        0.081|                 0.0|
|     1|   38|        0.152|               0.184|
|     2|  103|         0.26|               0.223|
|     3|  114|        0.353|               0.386|
|     4|  138|         0.45|               0.457|
|     5|  195|        0.551|               0.487|
|     6|  194|        0.647|               0.644|
|     7|  152|        0.751|               0.724|
|     8|  107|        0.843|               0.832|
|     9|   29|        0.924|               0.931|
+------+-----+-------------+--------------------+



## 3) Biggest upsets the model called right

Road underdogs (low predicted home-win prob) where the model correctly predicted the home
team would **lose** — sorted by how confident it was.

In [5]:
spark.sql(f"""
    SELECT p.game_date, g.away_team AS road_team, g.home_team AS host,
           g.away_points || '-' || g.home_points AS final_score,
           ROUND(p.pred_home_win_prob, 3) AS pred_home_win_prob
    FROM {PRED} p
    JOIN {GAMES} g ON p.game_sk = g.game_sk
    WHERE p.predicted_home_win = 0 AND p.correct = 1
    ORDER BY p.pred_home_win_prob ASC
    LIMIT 10
""").show(truncate=False)

print("Pipeline complete: Basketball Reference -> bronze -> silver -> gold -> model -> predictions.")

+----------+---------+----+-----------+------------------+
|game_date |road_team|host|final_score|pred_home_win_prob|
+----------+---------+----+-----------+------------------+
|2024-12-15|BOS      |WAS |112-98     |0.062             |
|2025-01-12|OKC      |WAS |136-95     |0.076             |
|2024-11-13|CLE      |PHI |114-106    |0.086             |
|2024-11-22|BOS      |WAS |108-96     |0.087             |
|2025-02-07|CLE      |WAS |134-124    |0.092             |
|2024-12-07|OKC      |NOP |119-109    |0.107             |
|2025-04-11|OKC      |UTA |145-111    |0.108             |
|2025-04-13|OKC      |NOP |115-100    |0.112             |
|2024-12-08|MEM      |WAS |140-112    |0.115             |
|2024-12-30|NYK      |WAS |126-106    |0.119             |
+----------+---------+----+-----------+------------------+

Pipeline complete: Basketball Reference -> bronze -> silver -> gold -> model -> predictions.


That's the full WAF demo end to end: ingested from Basketball Reference, refined through
bronze/silver/gold, modeled with MLflow, registered to Unity Catalog, and scored back into a
governed gold table. Re-run `07` to promote a new champion and `08` picks it up automatically.